# ECHOLOGY AI — Autonomous Pattern Discovery & Intelligence Engine

**A reproducible research notebook** for discovering patterns, investigating relationships, comparing competing hypotheses, building a lightweight knowledge graph, and simulating possible scenarios.

This notebook uses a public, built-in sample dataset so it runs without API keys. It is an experimental analytics system—not proof of causation or a production autonomous agent. Replace the sample data with your own CSV when ready.

**Pipeline:** Data intake → quality checks → exploratory analysis → anomaly detection → relationship graph → hypothesis council → scenario simulation → evidence-backed report.

## 0. Environment setup

The notebook uses common Python data-science packages. If an import fails, uncomment and run the installation cell.

In [ ]:
# Optional: install dependencies in a fresh environment
# %pip install -q pandas numpy matplotlib seaborn scikit-learn networkx scipy

In [ ]:
import warnings, json, math, textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("echology_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Environment ready. Outputs:", OUTPUT_DIR.resolve())

## 1. Signal Hunter — data intake

Load your own CSV by setting `DATA_PATH`. Otherwise, generate a reproducible synthetic event dataset representing activity across locations, categories and time. Synthetic data is for demonstrating the pipeline only.

In [ ]:
# Set to a CSV path to use your own data, e.g. DATA_PATH = "my_events.csv"
DATA_PATH = None

if DATA_PATH and Path(DATA_PATH).exists():
    df = pd.read_csv(DATA_PATH)
    data_origin = f"User-provided file: {DATA_PATH}"
else:
    rng = np.random.default_rng(RANDOM_STATE)
    n = 1200
    dates = pd.date_range("2025-01-01", periods=n, freq="h")
    location = rng.choice(["North", "South", "East", "West"], n)
    category = rng.choice(["mobility", "energy", "environment", "service"], n)
    hour = dates.hour.to_numpy()
    weekend = (dates.dayofweek >= 5).astype(int)
    location_effect = pd.Series(location).map({"North": 2.0, "South": -1.0, "East": 0.5, "West": 1.0}).to_numpy()
    category_effect = pd.Series(category).map({"mobility": 5.0, "energy": 3.0, "environment": -2.0, "service": 1.0}).to_numpy()
    daily_cycle = 4 * np.sin(2*np.pi*hour/24)
    seasonal = 2 * np.sin(2*np.pi*np.arange(n)/(24*30))
    noise = rng.normal(0, 2.0, n)
    metric_value = 30 + daily_cycle + seasonal + location_effect + category_effect - 2*weekend + noise
    # Inject a small number of unusual observations for anomaly exploration
    anomaly_idx = rng.choice(n, size=18, replace=False)
    metric_value[anomaly_idx] += rng.choice([-1, 1], len(anomaly_idx)) * rng.uniform(12, 22, len(anomaly_idx))
    df = pd.DataFrame({
        "timestamp": dates, "location": location, "category": category,
        "metric_value": metric_value, "volume": rng.poisson(12, n),
        "response_time": np.maximum(1, 18 - 0.18*metric_value + rng.normal(0, 2, n)),
        "weekend": weekend
    })
    data_origin = "Generated synthetic demonstration data"

df.head(), df.shape, data_origin

## 2. Data quality and preprocessing

Inspect missingness, duplicates, data types and basic ranges. The pipeline standardises timestamps where present and keeps a copy of the raw input.

In [ ]:
raw_df = df.copy()
quality = {
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_cells": int(df.isna().sum().sum()),
    "missing_by_column": df.isna().sum().to_dict(),
    "data_origin": data_origin
}
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.drop_duplicates().copy()
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
quality

## 3. Pattern Detective — exploratory analysis

Summarise distributions and trends. Choose a numeric target column to explore.

In [ ]:
TARGET = "metric_value" if "metric_value" in df.columns else (numeric_cols[0] if numeric_cols else None)
if TARGET is None:
    raise ValueError("Dataset needs at least one numeric column.")
print(df[TARGET].describe())
if "category" in df.columns:
    display(df.groupby("category")[TARGET].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False))
if "timestamp" in df.columns:
    daily = df.set_index("timestamp")[TARGET].resample("D").mean()
    plt.figure(figsize=(13,4))
    daily.plot()
    plt.title(f"Daily average {TARGET}")
    plt.xlabel("Date"); plt.ylabel(TARGET); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
plt.hist(df[TARGET], bins=35)
plt.title(f"Distribution of {TARGET}")
plt.xlabel(TARGET); plt.ylabel("Frequency"); plt.tight_layout(); plt.show()

## 4. Pattern Detective — correlation and anomaly screening

Correlation is descriptive, not causal. Isolation Forest flags unusual rows relative to the observed feature distribution; it does not establish that an event is harmful or erroneous.

In [ ]:
features = df.select_dtypes(include=np.number).copy()
if features.shape[1] >= 2:
    corr = features.corr(method="spearman")
    display(corr.style.background_gradient(cmap="coolwarm", axis=None))
    plt.figure(figsize=(8,6))
    plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    plt.colorbar(label="Spearman correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Feature correlation (descriptive)")
    plt.tight_layout(); plt.show()

X = features.fillna(features.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
df["anomaly_flag"] = (iso.fit_predict(X_scaled) == -1)
df["anomaly_score"] = -iso.score_samples(X_scaled)
print("Flagged rows:", int(df["anomaly_flag"].sum()), "of", len(df))
display(df.loc[df["anomaly_flag"]].head(10))

## 5. Relationship Explorer — build a lightweight knowledge graph

Create a graph from observed categorical co-occurrences. Nodes and edges are descriptive links from this dataset, not confirmed causal or biological relationships.

In [ ]:
G = nx.Graph()
categorical_cols = [c for c in ["location", "category"] if c in df.columns]
for col in categorical_cols:
    for value, count in df[col].value_counts().items():
        G.add_node(f"{col}:{value}", node_type=col, label=str(value), frequency=int(count))

if len(categorical_cols) >= 2:
    a, b = categorical_cols[:2]
    cross = df.groupby([a,b]).size()
    for (av,bv), count in cross.items():
        G.add_edge(f"{a}:{av}", f"{b}:{bv}", weight=int(count), relation="co-occurs")

print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())
if G.number_of_nodes():
    pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.8)
    plt.figure(figsize=(10,6))
    nx.draw_networkx(G, pos, with_labels=True, node_size=1600, font_size=8,
                     width=[max(0.5, G[u][v]["weight"]/8) for u,v in G.edges()])
    plt.title("Observed category co-occurrence graph")
    plt.axis("off"); plt.show()

centrality = nx.degree_centrality(G) if G.number_of_nodes() else {}
display(pd.DataFrame(centrality.items(), columns=["node","degree_centrality"]).sort_values("degree_centrality", ascending=False))

## 6. Agent Council — competing hypothesis templates

This section uses transparent, rule-based analyst roles rather than external autonomous LLM agents. It generates hypotheses from measured associations and records what evidence would be needed to test them.

In [ ]:
hypotheses = []
if "category" in df.columns and "location" in df.columns:
    grouped = df.groupby("category")[TARGET].mean().sort_values(ascending=False)
    if len(grouped) >= 2:
        top, bottom = grouped.index[0], grouped.index[-1]
        diff = float(grouped.iloc[0] - grouped.iloc[-1])
        hypotheses.append({
            "hypothesis": f"Average {TARGET} differs across categories ({top} vs {bottom}).",
            "supporting_observation": f"Observed mean difference: {diff:.3f}.",
            "skeptical_review": "Differences may reflect confounding, sampling imbalance or measurement effects.",
            "test_needed": "Use a pre-specified statistical test, adjust for confounders and validate on independent data."
        })

if "weekend" in df.columns:
    means = df.groupby("weekend")[TARGET].mean()
    if len(means) == 2:
        hypotheses.append({
            "hypothesis": f"Average {TARGET} differs between weekdays and weekends.",
            "supporting_observation": f"Weekday mean={means.loc[0]:.3f}; weekend mean={means.loc[1]:.3f}.",
            "skeptical_review": "Time-of-day and category composition may explain the difference.",
            "test_needed": "Fit a controlled model and assess uncertainty on a separate time period."
        })

council = pd.DataFrame(hypotheses)
display(council if not council.empty else "No hypothesis templates could be generated from available columns.")

## 7. Future Simulator — scenario analysis

A simple scenario model illustrates how a numeric metric might respond to changed inputs. It is not a forecast unless trained and validated on suitable historical data.

In [ ]:
scenario_results = {}
if "volume" in df.columns and TARGET in df.columns:
    X_model = df[["volume"]].astype(float)
    y_model = df[TARGET].astype(float)
    model = RandomForestRegressor(n_estimators=150, random_state=RANDOM_STATE, min_samples_leaf=5)
    model.fit(X_model, y_model)
    base_volume = float(df["volume"].median())
    scenarios = {
        "baseline": base_volume,
        "volume_minus_20pct": max(0, base_volume * 0.8),
        "volume_plus_20pct": base_volume * 1.2
    }
    for name, value in scenarios.items():
        scenario_results[name] = float(model.predict(pd.DataFrame({"volume":[value]}))[0])
    display(pd.DataFrame([{"scenario":k, "input_volume":v, f"estimated_{TARGET}":scenario_results[k]} for k,v in scenarios.items()]))
else:
    print("Scenario simulation requires a numeric 'volume' column.")

## 8. Discovery Reporter — evidence-backed summary

Generate a compact report from computed outputs. It deliberately separates observations, candidate hypotheses and limitations.

In [ ]:
report = {
    "project": "ECHOLOGY AI",
    "data_origin": data_origin,
    "dataset": {"rows_after_cleaning": int(len(df)), "columns": list(df.columns)},
    "quality": quality,
    "target_metric": TARGET,
    "anomalies": {
        "count": int(df["anomaly_flag"].sum()),
        "share": float(df["anomaly_flag"].mean()),
        "method": "IsolationForest",
        "interpretation": "Statistical screening flags; not verified incidents."
    },
    "knowledge_graph": {"nodes": G.number_of_nodes(), "edges": G.number_of_edges()},
    "candidate_hypotheses": hypotheses,
    "scenario_analysis": scenario_results,
    "limitations": [
        "Synthetic demonstration data unless a user CSV was supplied.",
        "Correlations and co-occurrences do not establish causation.",
        "Anomaly flags require domain review.",
        "Scenario outputs are illustrative and are not validated forecasts.",
        "Rule-based analyst roles are not autonomous LLM agents."
    ]
}
report_path = OUTPUT_DIR / "echology_intelligence_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)
df.to_csv(OUTPUT_DIR / "analysed_events.csv", index=False)
if not council.empty:
    council.to_csv(OUTPUT_DIR / "candidate_hypotheses.csv", index=False)
nx.write_graphml(G, OUTPUT_DIR / "relationship_graph.graphml")
print("Saved:", report_path)
print(json.dumps(report, indent=2, default=str)[:5000])

## 9. Next steps and responsible use

- Replace the synthetic data with a documented real dataset.
- Define domain-specific entities and validated relationships.
- Add statistical tests and uncertainty intervals before making claims.
- If adding LLM agents, require source citations, structured outputs and human review.
- Version datasets, parameters and model outputs for reproducibility.

**Interpretation rule:** ECHOLOGY AI discovers patterns worth investigating. It does not automatically prove causes, predict real-world events, or replace expert judgement.